In [0]:
from pyspark.sql import functions as F

root = "/Volumes/mvp_engdados/bronze/datafiles/"

def listar_csvs(caminho):
    arquivos = []
    for item in dbutils.fs.ls(caminho):
        if item.isDir():
            arquivos.extend(listar_csvs(item.path))
        elif item.name.lower().endswith(".csv"):
            arquivos.append(item.path)
    return arquivos

csv_paths = listar_csvs(root)

# Ajusta os prefixos dos arquivos para nomes padronizados de série.
mapa_series = {
    "brooklyn99": "brooklyn_99",
    "himym": "how_i_met_your_mother"
}

episodios = []
imdb = []

for path in csv_paths:
    nome_arquivo = path.rsplit("/", 1)[-1].lower()

    if nome_arquivo.endswith("_episodes.csv"):
        prefixo = nome_arquivo.removesuffix("_episodes.csv")
        destino = episodios
    elif nome_arquivo.endswith("_imdb.csv"):
        prefixo = nome_arquivo.removesuffix("_imdb.csv")
        destino = imdb
    else:
        continue

    serie = mapa_series.get(prefixo, prefixo)

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
        .withColumn("series_id", F.lit(serie))
        .withColumn("_source_file", F.lit(path))
        .withColumn("_ingested_at", F.current_timestamp())
    )

    destino.append(df)

def unir_dataframes(dataframes):
    resultado = dataframes[0]
    for df in dataframes[1:]:
        resultado = resultado.unionByName(df, allowMissingColumns=True)
    return resultado

episodios_bronze = unir_dataframes(episodios)
imdb_bronze = unir_dataframes(imdb)

In [0]:
display(
    episodios_bronze.groupBy("series_id")
    .count()
    .orderBy("series_id")
)

display(
    imdb_bronze.groupBy("series_id")
    .count()
    .orderBy("series_id")
)

series_id,count
big_bang_theory,279
breaking_bad,62
brooklyn_99,153
friends,236
game_of_thrones,73
how_i_met_your_mother,208
lost,121
the_office,201
the_walking_dead,177


series_id,count
big_bang_theory,280
breaking_bad,62
brooklyn_99,153
friends,235
game_of_thrones,73
how_i_met_your_mother,208
lost,119
the_office,188
the_walking_dead,177


In [0]:
(
    episodios_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.bronze.episodes_raw")
)

(
    imdb_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("mvp_engdados.bronze.imdb_raw")
)